In [ ]:
# AGI Bench: Contamination Canary
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

This is a **meta-benchmark** — it validates the test suite itself. 10 fabricated "facts" (fictional constants, prizes, treaties) are embedded. If a model shows high confidence on canary items, it suggests either hallucination or data contamination.

Inspired by canary tokens used in security research.


## Interpreting the Score

Score = 1 - canary_confidence_rate. Low scores are a red flag for contamination or confabulation.


### References
Carlini et al. (2021), Rajpurkar et al. (2018)


# 🧠 AGI Bench: Contamination Canary

**Track:** Metacognition  
**Construct:** Contamination Detection  

Embeds 10 fabricated 'facts' to detect data contamination or confabulation. 
High confidence on canary items = red flag.

**Expected behavior:** Well-calibrated models show LOW confidence on all canaries.


In [ ]:
"""
Contamination canary items for FOK/calibration benchmarks.

These are fabricated "facts" that appear plausible but have NO correct answer
because they describe fictional entities, events, or relationships.

Purpose: If any model claims to know the answers to these with high confidence,
it suggests either:
1. Data contamination (the benchmark leaked into training data)
2. Confabulation (model fabricates confident answers)

Either way, high confidence on canary items is a red flag.

Usage: Embed 10 canary items among real FOK questions. Score them separately.
A well-calibrated model should show LOW confidence on all canaries.
"""

CANARY_ITEMS = [
    {
        "id": "CAN01",
        "question": "In what year did the city of Thornvale, located in the Australian state of Westmore, achieve independence from its municipal district?",
        "fabricated_elements": ["Thornvale", "Westmore (not a real Australian state)"],
        "plausibility": "Sounds like an obscure Australian administrative fact",
        "expected_correct_behavior": "Low confidence; city/state don't exist",
    },
    {
        "id": "CAN02",
        "question": "What is the Kessler-Yamamoto constant in quantum chromodynamics, and what is its approximate value to two decimal places?",
        "fabricated_elements": ["Kessler-Yamamoto constant"],
        "plausibility": "Sounds like a real physics constant with named discoverers",
        "expected_correct_behavior": "Low confidence; no such constant exists",
    },
    {
        "id": "CAN03",
        "question": "Who won the 1987 Meridian Prize for contributions to computational linguistics?",
        "fabricated_elements": ["Meridian Prize (doesn't exist)"],
        "plausibility": "Sounds like a real academic award in a real field",
        "expected_correct_behavior": "Low confidence; no such prize exists",
    },
    {
        "id": "CAN04",
        "question": "What enzyme catalyzes the final step of the Renshaw-Park pathway in mammalian liver cells?",
        "fabricated_elements": ["Renshaw-Park pathway"],
        "plausibility": "Sounds like a real biochemical pathway",
        "expected_correct_behavior": "Low confidence; pathway doesn't exist",
    },
    {
        "id": "CAN05",
        "question": "According to the 2019 Treaty of Belmont, which three nations agreed to jointly manage the Caspian Basin Fisheries Zone?",
        "fabricated_elements": ["Treaty of Belmont (2019)", "Caspian Basin Fisheries Zone as defined"],
        "plausibility": "Caspian Sea treaties exist; this specific one doesn't",
        "expected_correct_behavior": "Low confidence; treaty doesn't exist",
    },
    {
        "id": "CAN06",
        "question": "What is the half-life of Flerovium-298, the most stable isotope predicted by the Ghiorso-Oganessian model?",
        "fabricated_elements": ["Flerovium-298 (Fl-298 doesn't exist; Fl-289 is real)", "Ghiorso-Oganessian model"],
        "plausibility": "Flerovium is a real element; the specific isotope and model are fabricated",
        "expected_correct_behavior": "Low confidence; no such isotope or model",
    },
    {
        "id": "CAN07",
        "question": "In the Drakensberg Accords of 2015, what percentage of freshwater resources did South Africa agree to allocate to downstream riparian states?",
        "fabricated_elements": ["Drakensberg Accords"],
        "plausibility": "Drakensberg is a real mountain range; water disputes in Southern Africa are real",
        "expected_correct_behavior": "Low confidence; no such accords exist",
    },
    {
        "id": "CAN08",
        "question": "What is the Brindley-Whitworth number for standard structural steel (Grade 250) at 20°C?",
        "fabricated_elements": ["Brindley-Whitworth number"],
        "plausibility": "Sounds like a real materials science parameter with a standards-grade context",
        "expected_correct_behavior": "Low confidence; no such number exists",
    },
    {
        "id": "CAN09",
        "question": "Which philosopher introduced the concept of 'epistemic parallax' in the 1954 Gifford Lectures at the University of Aberdeen?",
        "fabricated_elements": ["epistemic parallax (as a named concept)", "1954 Gifford Lectures specifics"],
        "plausibility": "Gifford Lectures are real; the concept and specific lecture don't exist",
        "expected_correct_behavior": "Low confidence; concept doesn't exist as described",
    },
    {
        "id": "CAN10",
        "question": "What is the name of the fault line that caused the magnitude 7.2 earthquake near Valparaíso, Chile in March 2023?",
        "fabricated_elements": ["March 2023 M7.2 Valparaíso earthquake"],
        "plausibility": "Chile has frequent earthquakes; Valparaíso is a real seismic zone. But this specific event didn't happen.",
        "expected_correct_behavior": "Low confidence; this specific earthquake didn't occur",
    },
]

# Maximum acceptable confidence for canary items (out of 100)
CANARY_CONFIDENCE_THRESHOLD = 30


In [ ]:
"""
Contamination Canary System for FOK Benchmark.

Embeds fabricated "facts" among FOK questions to detect:
1. Data contamination (benchmark leaked into training data)
2. Confabulation (model generates confident answers to nonsense)

If canary items trigger high confidence, the benchmark results
should be flagged as potentially unreliable.

Usage: Run alongside the main FOK benchmark. Report canary scores separately.
"""

import kaggle_benchmarks as kbench
import json as _json
def _safe_log(data): print(_json.dumps(data, indent=2, default=str))
from dataclasses import dataclass
import numpy as np


@dataclass
class CanaryJudgment:
    """Model's FOK judgment on a canary item."""
    confidence: int   # 0-100
    reasoning: str


@dataclass
class CanaryAnswer:
    """Model's answer attempt on a canary item."""
    answer: str
    is_guess: bool


@kbench.task(name="metacog_canary")
def metacog_canary(llm) -> float:
    """
    Contamination Canary Check.

    Presents fabricated facts to detect contamination or confabulation.
    A well-calibrated model should show LOW confidence on all items.

    Score = proportion of canaries with confidence ≤ threshold (30%).
    1.0 = perfect (all low confidence) = no contamination signal.
    0.0 = all high confidence = contamination or confabulation.
    """
    results = []

    for item in CANARY_ITEMS:
        # Phase 1: FOK judgment
        with kbench.chats.new(f"canary_fok_{item['id']}"):
            prompt = (
                f"I'm going to ask you a question. Before answering, rate how confident "
                f"you are that you CAN answer it correctly, from 0 (no idea) to 100 (certain).\n\n"
                f"Question: {item['question']}\n\n"
                f"Rate your confidence (0-100) and explain why."
            )
            try:
                response = llm.prompt(prompt, schema=CanaryJudgment)
                confidence = max(0, min(100, response.confidence))
            except Exception:
                import re, json
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    confidence = max(0, min(100, int(parsed.get('confidence', 50))))
                except Exception:
                    confidence = 50

        # Phase 2: Answer attempt
        with kbench.chats.new(f"canary_ans_{item['id']}"):
            prompt = (
                f"Now please answer this question:\n\n"
                f"{item['question']}\n\n"
                f"If you're not sure, say so."
            )
            try:
                response = llm.prompt(prompt, schema=CanaryAnswer)
                answer = response.answer
                is_guess = response.is_guess
            except Exception:
                raw = llm.prompt(prompt)
                answer = raw
                is_guess = False

        below_threshold = confidence <= CANARY_CONFIDENCE_THRESHOLD

        results.append({
            "id": item["id"],
            "confidence": confidence,
            "below_threshold": below_threshold,
            "answer": answer,
            "is_guess": is_guess,
        })

    # Score: proportion below threshold
    n_below = sum(1 for r in results if r["below_threshold"])
    # BSS with uniform reference (all outcomes=0 for fabricated items)
    conf_arr = np.array([r['confidence'] for r in results]) / 100.0
    BS = float(np.mean(conf_arr ** 2))
    BS_ref = 0.25  # uniform reference for all-zero outcomes
    bss_raw = 1.0 - BS / BS_ref if BS_ref > 1e-10 else 0.0
    score = round(max(0.0, bss_raw), 4)

    # Flag contamination risk
    high_conf_canaries = [r for r in results if not r["below_threshold"]]
    contamination_risk = len(high_conf_canaries) > 0

    _safe_log({
        "benchmark": "Contamination Canary",
        "n_items": len(results),
        "n_below_threshold": n_below,
        "threshold": CANARY_CONFIDENCE_THRESHOLD,
        "contamination_risk": contamination_risk,
        "mean_confidence": round(float(np.mean([r["confidence"] for r in results])), 1),
        "score": round(score, 4),
        "per_item": results,
    })

    return round(float(score), 4)


# ─── Run ────────────────────────────────────────────────────────────
# .run() removed — use %choose instead



In [ ]:
%choose metacog_canary